### Imports e Conexões

In [44]:
import os
import unicodedata
from dotenv import load_dotenv

# LangChain Core e Modelos
from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

# Ferramentas e o novo Agente (LangGraph)
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv()

# 1. Conectar ao Neo4j
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD")
)

# 2. Inicializar o LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0 
)
print("Infraestrutura conectada!")

Infraestrutura conectada!


### Esteira de Dados

In [45]:
cypher_template = """Você é um especialista em banco de dados Neo4j traduzindo perguntas em português para queries Cypher.

REGRAS OBRIGATÓRIAS:
1. Use APENAS os nós e relacionamentos fornecidos no schema.
2. NUNCA faça buscas textuais exatas usando '='. 
3. SEMPRE use a função apoc.text.clean() EM AMBOS OS LADOS da comparação com CONTAINS. 
4. CONTEXTO DE NEGÓCIO DA BASE DE DADOS:
   - Marcas, bancos, empresas e aplicativos ficam SEMPRE no nó Estabelecimento (e.nome).
   - Setores amplos, tipos de despesas e receitas ficam SEMPRE no nó Categoria (cat.nome).
   - Transações de saída de dinheiro têm t.tipo = 'Saída'. Entradas têm t.tipo = 'Entrada'.
5. Retorne APENAS o código Cypher válido, sem explicações.

Schema:
{schema}

Pergunta do usuário: {question}
Query Cypher:"""

cypher_prompt = PromptTemplate(input_variables=["schema", "question"], template=cypher_template)

chain_neo4j = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    cypher_prompt=cypher_prompt,
    verbose=True,
    allow_dangerous_requests=True 
)
print("Motor Cypher pronto!")

Motor Cypher pronto!


### Caixa de Ferramentas para o Langchain

In [46]:
@tool
def consultar_extrato(pergunta: str) -> str:
    """
    Use esta ferramenta SEMPRE que precisar consultar o histórico financeiro do usuário.
    IMPORTANTE: Repasse a pergunta do usuário EXATAMENTE como ele fez, sem alterar as palavras.
    """
    print(f"\n[Ferramenta Ativada] Consultando Neo4j para: '{pergunta}'")
    resposta = chain_neo4j.invoke({"query": pergunta})
    return resposta['result']

@tool
def simular_parcelamento(valor_total: float, parcelas: int) -> str:
    """
    Use esta ferramenta APENAS quando o usuário pedir para simular, planejar ou calcular 
    o parcelamento de uma dívida, compra ou fatura.
    """
    print(f"\n[Ferramenta Ativada] Calculando {parcelas}x de R$ {valor_total}")
    taxa = 0.03 # 3% de juros ao mês
    montante = valor_total * ((1 + taxa) ** parcelas)
    valor_parcela = montante / parcelas
    
    return f"O valor total com juros será R$ {montante:.2f}, dividido em {parcelas}x de R$ {valor_parcela:.2f}."

# Guardamos as ferramentas numa lista para dar para o Agente
ferramentas = [consultar_extrato, simular_parcelamento]
print("Caixa de Ferramentas montada!")

Caixa de Ferramentas montada!


### Cérebro do Agente

In [47]:
from langgraph.checkpoint.memory import MemorySaver

instrucoes_sistema = """Você é um Consultor Financeiro Inteligente do Itaú. 
Você tem ferramentas para consultar o banco de dados financeiro do usuário e simular parcelamentos. 
Sempre analise o que o usuário pediu e escolha a ferramenta correta. 

REGRAS DE FERRAMENTAS:
1. Se a resposta precisar de cálculo, NUNCA faça de cabeça, use a ferramenta de parcelamento.
2. NUNCA adicione comentários, tags ou textos como '<|channel|>commentary' ao nome da ferramenta. Use APENAS o nome exato.

Seja direto, profissional e prestativo."""

# 1. Criamos a memória RAM da sessão
memoria_do_agente = MemorySaver()

# 2. Injetamos a memória no Agente
agente_executor = create_react_agent(
    llm, 
    ferramentas, 
    prompt=instrucoes_sistema,
    checkpointer=memoria_do_agente
)
print("Agente ReAct com Memória inicializado com sucesso!")

Agente ReAct com Memória inicializado com sucesso!


/tmp/ipykernel_83982/2501438254.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_executor = create_react_agent(


### Loop para Simulação de um Chat

In [48]:
def limpar_texto(texto):
    texto_sem_acento = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto_sem_acento.lower()

print("Consultor Financeiro Proativo! (Digite 'sair' para encerrar)\n" + "-"*50)

# Configuramos o ID da conversa atual
config_sessao = {"configurable": {"thread_id": "sessao_luis_01"}}

while True:
    pergunta_crua = input("Você: ")
    
    if pergunta_crua.lower() in ['sair', 'exit', 'quit']:
        print("Encerrando o chat. Até a próxima!")
        break
        
    pergunta_limpa = limpar_texto(pergunta_crua)

    print("Analisando intenção e histórico...")
    
    # Invocamos o Agente passando a pergunta E a configuração da sessão
    resposta = agente_executor.invoke(
        {"messages": [("human", pergunta_limpa)]}, 
        config=config_sessao
    )
    
    resultado_ia = resposta["messages"][-1].content
    
    print(f"\nIA: {resultado_ia}\n" + "="*50)

Consultor Financeiro Proativo! (Digite 'sair' para encerrar)
--------------------------------------------------
Analisando intenção e histórico...

[Ferramenta Ativada] Consultando Neo4j para: 'quanto gastei no uber?'


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Cliente)-[:REALIZOU]->(t:Transacao)-[:NO_ESTABELECIMENTO]->(e:Estabelecimento)
WHERE t.tipo = 'Saída' AND apoc.text.clean(e.nome) CONTAINS apoc.text.clean('Uber')
RETURN sum(t.valor) AS totalUber;
Full Context:
[{'totalUber': 132.3}]

> Finished chain.

IA: Você gastou R$ 132,30 no Uber.
Analisando intenção e histórico...

[Ferramenta Ativada] Consultando Neo4j para: 'e no ifood?'


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Cliente)-[:REALIZOU]->(t:Transacao)-[:NO_ESTABELECIMENTO]->(e:Estabelecimento)
WHERE apoc.text.clean(e.nome) CONTAINS apoc.text.clean('ifood')
RETURN c.id AS clienteId, t.data AS data, t.valor AS valor, t.tipo AS tipo, e.nome AS estabelecimento;
Full Con